In [ ]:
# 1. 数据准备：读取类别 + 动态生成 8000 张抽样清单
import json
import random
import yaml
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/wangsiyi1017/traffic/TrafficDetYolo")
DATA_YAML = str(DATA_ROOT / "data.yaml")

names = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))["names"]
json.dump(names, open("/kaggle/working/traffic_classes.json", "w"), ensure_ascii=False, indent=2)
print("✅ 数据就绪:", DATA_ROOT)
print("类别列表:", names)

train_img_dir = DATA_ROOT / "train" / "images"
all_imgs = [f.name for f in train_img_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
random.seed(42)
sample_imgs = random.sample(all_imgs, min(8000, len(all_imgs)))

sample_txt_path = "/kaggle/working/train_sample_v11n.txt"
with open(sample_txt_path, "w") as f:
    for img in sample_imgs:
        f.write(f"{DATA_ROOT}/train/images/{img}\n")

data = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
data["path"] = str(DATA_ROOT)
data["train"] = sample_txt_path
data["val"] = "val/images"

SAMPLE_YAML = "/kaggle/working/traffic_sample_v11n.yaml"
with open(SAMPLE_YAML, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(f"\n✅ 抽样配置生成完毕！图片数: {len(sample_imgs)} 张")
print(f"配置路径: {SAMPLE_YAML}")

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    print("正在安装 ultralytics...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

from ultralytics import YOLO
model = YOLO("yolo11n.pt")  # 自动下载约 5.4MB
print("✅ 环境准备完毕，yolo11n.pt 已就绪")

In [ ]:
# 3. 消融实验训练
from ultralytics import YOLO

EPOCHS, IMGSZ, BATCH, SEED = 5, 320, 4, 42
PROJECT_DIR = "/kaggle/working/traffic_models_v11n"

RUNS = [
    dict(name="v11n_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),
    dict(name="v11n_lr0.005",                   lr=0.005, wd=0.0005),
    dict(name="v11n_lr0.001",                   lr=0.001, wd=0.0005),
]
BEST_LR = 0.01
for _wd in (0.001, 0.0001):
    RUNS.append(dict(name=f"v11n_wd{_wd}", lr=BEST_LR, wd=_wd))

for run in RUNS:
    print(f"\n{'='*60}")
    print(f"🚀 开始训练: {run['name']} (lr={run['lr']}, wd={run['wd']})")
    print(f"{'='*60}\n")

    model = YOLO("yolo11n.pt")
    model.train(
        data="/kaggle/working/traffic_sample_v11n.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        optimizer='SGD',
        lr0=run['lr'],
        lrf=1.0,
        weight_decay=run['wd'],
        project=PROJECT_DIR,
        name=run['name'],
        exist_ok=True,
        device=0,
        workers=2,
        cache=False,
        verbose=False
    )

print("\n✅ 全部 YOLOv11n 消融实验训练完成！")

In [ ]:
# 4. 汇总各次实验指标
import pandas as pd
import json
from pathlib import Path

RUN_DIR = Path("/kaggle/working/traffic_models_v11n")
RUNS = [
    dict(name="v11n_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),
    dict(name="v11n_lr0.005",                   lr=0.005, wd=0.0005),
    dict(name="v11n_lr0.001",                   lr=0.001, wd=0.0005),
    dict(name="v11n_wd0.001",                   lr=0.01,  wd=0.001),
    dict(name="v11n_wd0.0001",                  lr=0.01,  wd=0.0001),
]

def pick_col(df, *needles):
    for c in df.columns:
        if all(k in c for k in needles):
            return c
    raise KeyError(f"未找到包含 {needles} 的列")

def parse_run(name, lr, wd):
    csv_path = RUN_DIR / name / "results.csv"
    if not csv_path.exists():
        print(f"⚠️ 跳过 {name}: 未找到结果文件")
        return None
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    if "epoch" not in df.columns:
        df.insert(0, "epoch", range(len(df)))
    try:
        c_ep = pick_col(df, "epoch")
        c_p = pick_col(df, "precision")
        c_r = pick_col(df, "recall")
        c_m50 = pick_col(df, "mAP50")
        c_m5095 = pick_col(df, "mAP50-95")
    except KeyError as e:
        print(f"⚠️ 解析 {name} 失败: {e}")
        return None

    best_i = df[c_m50].idxmax()
    best = df.loc[best_i]
    return dict(name=name, lr=lr, wd=wd, epochs=int(best[c_ep]) + 1,
                best_epoch=int(best[c_ep]) + 1,
                precision=float(best[c_p]), recall=float(best[c_r]),
                mAP50=float(best[c_m50]), mAP5095=float(best[c_m5095]),
                curves=dict(epoch=df[c_ep].astype(int).tolist(),
                            mAP50=df[c_m50].tolist(),
                            mAP5095=df[c_m5095].tolist()))

summaries = [s for s in (parse_run(**r) for r in RUNS) if s is not None]

if summaries:
    tbl = pd.DataFrame(summaries)[["name", "lr", "wd", "precision", "recall", "mAP50", "mAP5095"]]
    print("📊 YOLOv11n 消融实验结果汇总（按 mAP50 降序排列）")
    print("="*60)
    print(tbl.sort_values("mAP50", ascending=False).to_string(index=False))
    json.dump(summaries, open("/kaggle/working/yolov11n_runs_summary.json", "w"), ensure_ascii=False, indent=2)
    best = max(summaries, key=lambda s: s["mAP50"])
    print(f"\n🏆 表现最好的参数组合: {best['name']}")
    print(f"👉 最优 lr = {best['lr']}, 最优 wd = {best['wd']}, 最高 mAP50 = {best['mAP50']:.4f}")
else:
    print("❌ 没有找到任何有效实验结果")

In [ ]:
# 5. 选出最优实验并导出权重
import shutil, pathlib

best = max(summaries, key=lambda s: s["mAP50"])
src = pathlib.Path("/kaggle/working/traffic_models_v11n") / best["name"] / "weights/best.pt"
shutil.copy2(src, "/kaggle/working/yolo11n_traffic_best.pt")
print(f"✅ 已导出测试权重: {best['name']} (mAP50={best['mAP50']:.4f})")

In [ ]:
# 6. 绘制消融对比曲线
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for s in summaries:
    plt.plot(s["curves"]["epoch"], s["curves"]["mAP50"], marker="o", label=s["name"])
plt.xlabel("epoch"); plt.ylabel("mAP@0.5"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title("YOLOv11n ablation: mAP@0.5")
plt.tight_layout()
plt.savefig("/kaggle/working/yolov11n_ablation_curves.png", dpi=150)
plt.show()

In [ ]:
# 7. 生成全量训练的绝对路径 yaml
import yaml
from pathlib import Path

DATA_ROOT = "/kaggle/input/datasets/wangsiyi1017/traffic/TrafficDetYolo"
with open(Path(DATA_ROOT) / "data.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
data["path"] = DATA_ROOT
data["train"] = "train/images"
data["val"] = "val/images"

NEW_YAML = "/kaggle/working/traffic_data_abs_v11n.yaml"
with open(NEW_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data, f, sort_keys=False)
print(f"✅ 全量训练 yaml 已生成: {NEW_YAML}")

In [ ]:
# 8. 全量数据最终训练
from ultralytics import YOLO

BEST_LR = 0.001   
BEST_WD = 0.0005   

model = YOLO("yolo11n.pt")
model.train(
    data="/kaggle/working/traffic_data_abs_v11n.yaml",
    imgsz=640,
    epochs=20,
    patience=10,
    optimizer='SGD',
    lr0=BEST_LR,
    lrf=0.1,
    weight_decay=BEST_WD,
    batch=16,          # nano 模型显存占用小，batch 可以设大
    seed=42,
    project="/kaggle/working/traffic_models_v11n_full",
    name="yolo11n_full_best",
    exist_ok=True,
    device=0,
    workers=2,
    cache=False
)

In [ ]:
# 9. 打包全量训练结果
import shutil, os
from pathlib import Path

RESULT_DIR = Path("/kaggle/working/traffic_models_v11n_full/yolo11n_full_best")
ZIP_PATH = "/kaggle/working/yolo11n_full_best_results.zip"

if not RESULT_DIR.exists():
    print(f"❌ 找不到结果目录: {RESULT_DIR}")
else:
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    shutil.make_archive(base_name=ZIP_PATH.replace(".zip", ""), format='zip', root_dir=RESULT_DIR)
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"✅ 打包完成: {ZIP_PATH} ({zip_size_mb:.1f} MB)")

In [ ]:
# 10. 四模型横向对比（如果你已经下载了前面四个模型的 results.csv）
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ⚠️ 请根据实际情况修改文件名和路径
csv_files = {
    "YOLOv8s":  "/kaggle/input/datasets/wangsiyi1017/yolo-train-data/yolo_compare/v8_results.csv",
    "YOLOv8n":  "/kaggle/input/datasets/wangsiyi1017/yolo-train-data/yolo_compare/8n_results.csv",
    "YOLOv11s": "/kaggle/input/datasets/wangsiyi1017/yolo-train-data/yolo_compare/v11_results.csv",
    "YOLOv11n": "/kaggle/working/traffic_models_v11n_full/yolo11n_full_best/results.csv",
}

rows = []
for name, path in csv_files.items():
    if not Path(path).exists():
        print(f"⚠️ 跳过 {name}: 找不到 {path}")
        continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    mAP50_col = next(c for c in df.columns if "mAP50" in c and "95" not in c)
    best = df.loc[df[mAP50_col].idxmax()]
    rows.append(dict(model=name, mAP50=round(float(best[mAP50_col]), 4)))

if rows:
    df_cmp = pd.DataFrame(rows)
    print("📊 四模型 mAP@0.5 对比")
    print(df_cmp.to_string(index=False))

    plt.figure(figsize=(9, 5))
    colors = ["#7c3aed", "#06b6d4", "#f59e0b", "#10b981"]
    bars = plt.bar(df_cmp["model"], df_cmp["mAP50"], color=colors[:len(df_cmp)])
    for bar, v in zip(bars, df_cmp["mAP50"]):
        plt.text(bar.get_x() + bar.get_width()/2, v, f"{v:.4f}",
                 ha="center", va="bottom", fontweight="bold")
    plt.ylabel("mAP@0.5"); plt.title("YOLO Model Comparison (Full Data)")
    plt.grid(alpha=.3, axis="y")
    plt.tight_layout()
    plt.savefig("/kaggle/working/four_model_comparison.png", dpi=150)
    plt.show()
else:
    print("❌ 没有找到任何模型结果文件")